# MMLU-Redux 2.0 벤치마크 + 배치 처리 + 토큰당 추론 시간

양자화된 EXAONE 4.0 1.2B 모델을 **MMLU-Redux 2.0** 데이터셋으로 평가합니다.

## 추론 전략
| 방식 | 설명 | 속도 |
|------|------|------|
| **logit (배치)** | 마지막 토큰의 A/B/C/D logit 비교, 배치로 묶어 처리 | ⚡ 빠름 |
| **generation (배치)** | `generate()` 실제 토큰 생성 - vLLM 서빙 유사 | 🐢 느림 |

기본값은 **logit 배치 방식** (정확도는 동일, 속도 우선).
필요 시 `INFERENCE_MODE = "generation"` 으로 전환 가능.

In [ ]:
import os
import json
import time
import zipfile
import shutil
import tempfile
from datetime import datetime
from pathlib import Path

import torch
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from datasets import load_dataset, get_dataset_config_names
from transformers import AutoModelForCausalLM, AutoTokenizer

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU      : {gpu}")
    print(f"VRAM     : {vram:.1f} GB")

## ⚙️ 설정

- `BATCH_SIZE`: 한 번에 처리할 샘플 수. VRAM에 따라 조정
  - RTX 4060 8GB + 1.9GB 모델 → **8~16** 권장
  - OOM 발생 시 줄이세요
- `INFERENCE_MODE`: `"logit"` (빠름) / `"generation"` (vLLM 유사)
- `TIMING_SAMPLES`: 타이밍 측정 샘플 수 (`None` = 전체)

In [9]:
# ── 모델 소스 ───────────────────────────────────
MODEL_SOURCE = "zip"                          # "dir" or "zip"
MODEL_DIR    = "./model"
ZIP_PATH     = "./baseline_submit.zip"
ZIP_SUBDIR   = "model"

# ── 평가 설정 ───────────────────────────────────
DATASET_ID  = "edinburgh-dawg/mmlu-redux-2.0"
SPLIT       = "test"
MAX_SAMPLES = None        # None=전체, 정수=샘플 제한
SUBJECTS    = None        # None=전체

# ── 추론 설정 ────────────────────────────────────
INFERENCE_MODE  = "logit"   # "logit" (빠름) | "generation" (vLLM 유사)
BATCH_SIZE      = 8         # 배치 크기 (OOM 시 줄이세요)
MAX_NEW_TOKENS  = 8         # generation 모드 전용
TIMING_SAMPLES  = None      # None=전체 측정, 정수=앞 N개만 측정
WARMUP_RUNS     = 3

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.float16 if torch.cuda.is_available() else torch.float32

OUTPUT_DIR = "./mmlu_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Inference mode  : {INFERENCE_MODE}")
print(f"Batch size      : {BATCH_SIZE}")
print(f"Device / Dtype  : {DEVICE} / {DTYPE}")
print(f"Dataset         : {DATASET_ID} [{SPLIT}]")
print(f"Timing samples  : {TIMING_SAMPLES if TIMING_SAMPLES else 'ALL'}")

Inference mode  : logit
Batch size      : 8
Device / Dtype  : cuda / torch.float16
Dataset         : edinburgh-dawg/mmlu-redux-2.0 [test]
Timing samples  : ALL


## 📦 모델 로드

In [10]:
_tmp_dir = None

if MODEL_SOURCE == "zip":
    print(f"[INFO] {ZIP_PATH} 압축 해제 중...")
    _tmp_dir = tempfile.mkdtemp(prefix="mmlu_eval_")
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(_tmp_dir)
    model_path = os.path.join(_tmp_dir, ZIP_SUBDIR)
elif MODEL_SOURCE == "dir":
    model_path = MODEL_DIR
else:
    raise ValueError("MODEL_SOURCE는 'dir' 또는 'zip'")

assert os.path.exists(model_path), f"모델 경로 없음: {model_path}"
print(f"[INFO] 모델 경로: {os.path.abspath(model_path)}")
for f in sorted(os.listdir(model_path)):
    size = os.path.getsize(os.path.join(model_path, f)) / 1e6
    print(f"  {f:40s}  {size:8.1f} MB")

[INFO] ./baseline_submit.zip 압축 해제 중...
[INFO] 모델 경로: C:\Users\sinja\AppData\Local\Temp\mmlu_eval_u0xdvcbq\model
  chat_template.jinja                            0.0 MB
  config.json                                    0.0 MB
  generation_config.json                         0.0 MB
  merges.txt                                     1.2 MB
  model.safetensors                           1390.7 MB
  recipe.yaml                                    0.0 MB
  special_tokens_map.json                        0.0 MB
  tokenizer.json                                 7.9 MB
  tokenizer_config.json                          0.1 MB
  vocab.json                                     1.9 MB


In [11]:
print("[INFO] 토크나이저 로드 중...")
tokenizer = AutoTokenizer.from_pretrained(
    model_path, trust_remote_code=True,
)
tokenizer.padding_side = "left"   # 배치 처리에 중요 (decoder-only)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("[INFO] 모델 로드 중...")
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=DTYPE,
    device_map=DEVICE,
    trust_remote_code=True,
)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"[INFO] 모델 파라미터: {n_params / 1e9:.3f}B")
if DEVICE == "cuda":
    used = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"[INFO] VRAM 사용: {used:.2f} GB / {total:.1f} GB")

[INFO] 토크나이저 로드 중...


The tokenizer you are loading from 'C:\Users\sinja\AppData\Local\Temp\mmlu_eval_u0xdvcbq\model' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


[INFO] 모델 로드 중...


Compressing model: 210it [00:00, 1043.61it/s]


[INFO] 모델 파라미터: 0.352B
[INFO] VRAM 사용: 3.55 GB / 8.6 GB


## 📊 MMLU-Redux 2.0 데이터셋 로드

In [12]:
print(f"[INFO] 데이터셋 조회: {DATASET_ID}")

configs = []
try:
    configs = get_dataset_config_names(DATASET_ID)
    configs = [c for c in configs if c != "default"]
    print(f"[INFO] config {len(configs)}개: {configs[:5]}...")
except Exception as e:
    print(f"[INFO] config 조회 실패: {e}")

all_records = []

if configs:
    target_configs = configs if SUBJECTS is None else [c for c in configs if c in SUBJECTS]
    for cfg in tqdm(target_configs, desc="Loading subjects"):
        try:
            ds = load_dataset(DATASET_ID, cfg, split=SPLIT)
            for row in ds:
                choices = row.get("choices", row.get("options", []))
                answer  = row.get("answer", row.get("correct_answer", row.get("label", 0)))
                all_records.append({"subject": cfg, "question": row["question"],
                                    "choices": choices, "answer": answer})
        except Exception:
            pass

if not all_records:
    for try_split in [SPLIT, "test", "validation", "train"]:
        try:
            ds_all = load_dataset(DATASET_ID, split=try_split)
            for row in ds_all:
                subject = row.get("subject", row.get("category", "unknown"))
                choices = row.get("choices", row.get("options", []))
                answer  = row.get("answer", row.get("label", 0))
                all_records.append({"subject": subject, "question": row["question"],
                                    "choices": choices, "answer": answer})
            break
        except Exception:
            continue

assert all_records, "[ERROR] 레코드 0건"

df = pd.DataFrame(all_records)

def normalize_answer(ans):
    if isinstance(ans, int): return ans
    if isinstance(ans, str):
        m = {"A": 0, "B": 1, "C": 2, "D": 3}
        v = ans.strip().upper()
        if v in m: return m[v]
        if v.isdigit(): return int(v)
    try: return int(ans)
    except: return 0

df["answer"] = df["answer"].apply(normalize_answer)
before = len(df)
df = df[df["choices"].apply(lambda x: isinstance(x, list) and len(x) == 4)].reset_index(drop=True)
if len(df) < before:
    print(f"[WARN] choices 불량 {before - len(df)}건 제외")

if MAX_SAMPLES is not None:
    df = df.sample(min(MAX_SAMPLES, len(df)), random_state=42).reset_index(drop=True)

print(f"\n[OK] 데이터셋 준비: {len(df)}건 / {df['subject'].nunique()}개 subject")

[INFO] 데이터셋 조회: edinburgh-dawg/mmlu-redux-2.0
[INFO] config 57개: ['abstract_algebra', 'anatomy', 'astronomy', 'business_ethics', 'clinical_knowledge']...


Loading subjects: 100%|██████████| 57/57 [01:34<00:00,  1.66s/it]


[OK] 데이터셋 준비: 5700건 / 57개 subject


## 🔍 배치 추론 함수

### logit 방식 (배치)
- 각 샘플의 프롬프트를 **배치로 묶어** 한 번에 forward
- 마지막 토큰 위치의 A/B/C/D logit 비교 → 단일 샘플 대비 **BATCH_SIZE배 빠름**

### generation 방식 (배치)
- `model.generate()`를 배치로 호출 (vLLM 서빙 유사)
- 동일하게 배치 처리하여 throughput 향상

In [13]:
CHOICE_LETTERS = ["A", "B", "C", "D"]

# A/B/C/D 단일 토큰 ID 탐색
choice_token_ids = []
for letter in CHOICE_LETTERS:
    found = False
    for variant in [f" {letter}", letter, f"({letter})", f"{letter}."]:
        ids = tokenizer.encode(variant, add_special_tokens=False)
        if len(ids) == 1:
            choice_token_ids.append(ids[0])
            found = True
            break
    if not found:
        choice_token_ids.append(tokenizer.encode(letter, add_special_tokens=False)[0])

choice_tid_set = set(choice_token_ids)
print(f"[INFO] 선택지 토큰 ID: {dict(zip(CHOICE_LETTERS, choice_token_ids))}")


def build_prompt(question: str, choices: list) -> str:
    """EXAONE chat template 기반 프롬프트"""
    content = "다음 객관식 문제를 읽고 정답 알파벳(A, B, C, D 중 하나)만 출력하세요.\n\n"
    content += f"문제: {question}\n"
    for letter, choice in zip(CHOICE_LETTERS, choices):
        content += f"{letter}. {choice}\n"
    content += "\n정답:"
    messages = [{"role": "user", "content": content}]
    try:
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True)
    except Exception:
        return content


# ──────────────────────────────────────────────────────
#  [방식 1] logit 배치 추론
# ──────────────────────────────────────────────────────
@torch.no_grad()
def batch_predict_logit(batch_questions, batch_choices):
    """
    여러 샘플을 한 번에 forward → 각 샘플 마지막 유효 토큰의 logit 비교.
    반환: (predictions list, elapsed_ms float, n_tokens int)
    """
    prompts = [build_prompt(q, c) for q, c in zip(batch_questions, batch_choices)]

    enc = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=2048,
    ).to(DEVICE)

    n_input_tokens = enc["input_ids"].numel()  # timing용

    if DEVICE == "cuda": torch.cuda.synchronize()
    t0 = time.perf_counter()

    outputs = model(**enc)

    if DEVICE == "cuda": torch.cuda.synchronize()
    elapsed_ms = (time.perf_counter() - t0) * 1000

    logits = outputs.logits  # (B, seq_len, vocab)
    attention_mask = enc["attention_mask"]  # (B, seq_len)

    preds = []
    for i in range(logits.shape[0]):
        # 패딩을 제외한 마지막 실제 토큰 위치
        last_pos = attention_mask[i].nonzero(as_tuple=False)[-1].item()
        token_logits = logits[i, last_pos, :]  # (vocab,)
        scores = [token_logits[tid].item() for tid in choice_token_ids]
        preds.append(int(np.argmax(scores)))

    return preds, elapsed_ms, n_input_tokens


# ──────────────────────────────────────────────────────
#  [방식 2] generation 배치 추론 (vLLM 서빙 유사)
# ──────────────────────────────────────────────────────
@torch.no_grad()
def batch_predict_generation(batch_questions, batch_choices):
    """
    model.generate() 배치 호출 → greedy decoding.
    반환: (predictions list, elapsed_ms float, n_new_tokens int)
    """
    prompts = [build_prompt(q, c) for q, c in zip(batch_questions, batch_choices)]

    enc = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=2048,
    ).to(DEVICE)

    input_len = enc["input_ids"].shape[1]

    if DEVICE == "cuda": torch.cuda.synchronize()
    t0 = time.perf_counter()

    out = model.generate(
        **enc,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        temperature=1.0,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
        use_cache=True,
    )

    if DEVICE == "cuda": torch.cuda.synchronize()
    elapsed_ms = (time.perf_counter() - t0) * 1000

    n_new_tokens = (out.shape[1] - input_len) * out.shape[0]

    preds = []
    for i in range(out.shape[0]):
        new_ids = out[i, input_len:].tolist()
        pred = -1
        for tid in new_ids:
            if tid in choice_tid_set:
                pred = choice_token_ids.index(tid)
                break
        if pred == -1:
            text = tokenizer.decode(new_ids, skip_special_tokens=True).strip().upper()
            for j, letter in enumerate(CHOICE_LETTERS):
                if text.startswith(letter):
                    pred = j
                    break
        preds.append(pred)

    return preds, elapsed_ms, n_new_tokens


# 선택된 방식에 따라 함수 연결
if INFERENCE_MODE == "logit":
    batch_predict = batch_predict_logit
    print("[INFO] 추론 방식: logit 배치 (빠름)")
else:
    batch_predict = batch_predict_generation
    print("[INFO] 추론 방식: generation 배치 (vLLM 유사)")

[INFO] 선택지 토큰 ID: {'A': 780, 'B': 876, 'C': 784, 'D': 903}
[INFO] 추론 방식: logit 배치 (빠름)


In [14]:
print(f"[INFO] GPU 워밍업 ({WARMUP_RUNS}회)...")

warmup_rows = df.head(min(BATCH_SIZE, len(df)))
with torch.no_grad():
    for _ in range(WARMUP_RUNS):
        batch_predict(
            warmup_rows["question"].tolist(),
            warmup_rows["choices"].tolist(),
        )

if DEVICE == "cuda":
    torch.cuda.synchronize()
    used = torch.cuda.memory_allocated() / 1e9
    peak = torch.cuda.max_memory_allocated() / 1e9
    print(f"[INFO] VRAM 현재: {used:.2f} GB / 피크: {peak:.2f} GB")

print("[INFO] 워밍업 완료")

[INFO] GPU 워밍업 (3회)...
[INFO] VRAM 현재: 5.69 GB / 피크: 6.08 GB
[INFO] 워밍업 완료


## 🚀 배치 평가 실행

In [15]:
all_preds      = []
latencies_ms   = []   # 배치 단위 지연 시간
token_counts   = []   # 배치 단위 처리 토큰 수
errors         = []

timing_limit = len(df) if TIMING_SAMPLES is None else TIMING_SAMPLES
measured_so_far = 0

n_batches = (len(df) + BATCH_SIZE - 1) // BATCH_SIZE

eval_start = time.perf_counter()

for batch_idx in tqdm(range(n_batches), desc=f"배치 평가 (bs={BATCH_SIZE})"):
    start = batch_idx * BATCH_SIZE
    end   = min(start + BATCH_SIZE, len(df))
    batch = df.iloc[start:end]

    try:
        preds, elapsed_ms, n_tokens = batch_predict(
            batch["question"].tolist(),
            batch["choices"].tolist(),
        )
        all_preds.extend(preds)

        if measured_so_far < timing_limit:
            latencies_ms.append(elapsed_ms)
            token_counts.append(n_tokens)
            measured_so_far += len(batch)

    except torch.cuda.OutOfMemoryError:
        print(f"\n[ERROR] OOM at batch {batch_idx}. BATCH_SIZE를 줄이세요.")
        torch.cuda.empty_cache()
        all_preds.extend([-1] * len(batch))
    except Exception as e:
        errors.append((batch_idx, str(e)))
        all_preds.extend([-1] * len(batch))

total_eval_sec = time.perf_counter() - eval_start

df["prediction"] = all_preds
df["correct"]    = df["prediction"] == df["answer"]

if errors:
    print(f"[WARN] 배치 오류 {len(errors)}건")

valid_df    = df[df["prediction"] >= 0]
overall_acc = valid_df["correct"].mean() * 100

print(f"\n{'='*52}")
print(f"  전체 정확도: {overall_acc:.2f}%  ({int(valid_df['correct'].sum())}/{len(valid_df)})")
print(f"  총 평가 시간: {total_eval_sec:.1f}초 ({total_eval_sec/60:.1f}분)")
print(f"  평균 처리 속도: {len(df)/total_eval_sec:.1f} samples/sec")
print(f"{'='*52}")

배치 평가 (bs=8):  24%|██▍       | 170/713 [18:19<58:31,  6.47s/it]  


KeyboardInterrupt: 

## ⏱️ 토큰당 추론 시간 분석

In [ ]:
latencies_ms = np.array(latencies_ms)
token_counts = np.array(token_counts)

# 배치당 총 토큰 수 / 배치 지연 → 토큰당 처리 속도
ms_per_token   = latencies_ms / np.maximum(token_counts, 1)
tokens_per_sec = token_counts / (latencies_ms / 1000 + 1e-9)

# 샘플당 평균 latency
actual_batch_sizes = np.minimum(
    np.arange(1, len(latencies_ms) + 1) * BATCH_SIZE, len(df)
) - np.maximum(np.arange(len(latencies_ms)) * BATCH_SIZE, 0)
ms_per_sample = latencies_ms / np.maximum(actual_batch_sizes, 1)

mode_label = "(logit/input tokens)" if INFERENCE_MODE == "logit" else "(generation/output tokens)"

print(f"\n{'='*56}")
print(f"  ⏱️  추론 속도 측정 결과  {mode_label}")
print(f"  배치 크기: {BATCH_SIZE}  /  측정 배치 수: {len(latencies_ms)}")
print(f"{'='*56}")
print(f"  [배치 처리 시간 (ms/batch)]")
print(f"    평균:   {latencies_ms.mean():.1f} ms")
print(f"    P50:    {np.median(latencies_ms):.1f} ms")
print(f"    P90:    {np.percentile(latencies_ms, 90):.1f} ms")
print(f"    P99:    {np.percentile(latencies_ms, 99):.1f} ms")
print()
print(f"  [샘플당 추론 시간 (ms/sample)]")
print(f"    평균:   {ms_per_sample.mean():.1f} ms")
print(f"    P50:    {np.median(ms_per_sample):.1f} ms")
print(f"    P90:    {np.percentile(ms_per_sample, 90):.1f} ms")
print()
print(f"  [토큰당 처리 시간 (ms/token)]")
print(f"    평균:   {ms_per_token.mean():.3f} ms/token")
print(f"    P50:    {np.median(ms_per_token):.3f} ms/token")
print()
print(f"  [처리량]")
print(f"    평균:   {tokens_per_sec.mean():.0f} tokens/sec")
print(f"    P50:    {np.median(tokens_per_sec):.0f} tokens/sec")
print(f"  전체 샘플 속도: {len(df)/total_eval_sec:.1f} samples/sec")
print(f"{'='*56}")

## 📈 Subject별 정확도

In [ ]:
subject_acc = (
    valid_df.groupby("subject")["correct"]
    .agg(["mean", "count", "sum"])
    .rename(columns={"mean": "accuracy", "count": "total", "sum": "correct_cnt"})
    .sort_values("accuracy", ascending=False)
)
subject_acc["accuracy"] = (subject_acc["accuracy"] * 100).round(2)

print("== 상위 20개 Subject ==")
print(subject_acc.head(20).to_string())
print("\n== 하위 10개 Subject ==")
print(subject_acc.tail(10).to_string())

## 💾 결과 저장

In [ ]:
timing_stats = {
    "inference_mode"           : INFERENCE_MODE,
    "batch_size"               : BATCH_SIZE,
    "measured_batches"         : int(len(latencies_ms)),
    "total_eval_sec"           : round(total_eval_sec, 2),
    "samples_per_sec"          : round(len(df) / total_eval_sec, 2),
    "batch_latency_mean_ms"    : round(float(latencies_ms.mean()), 2),
    "batch_latency_p50_ms"     : round(float(np.median(latencies_ms)), 2),
    "batch_latency_p90_ms"     : round(float(np.percentile(latencies_ms, 90)), 2),
    "batch_latency_p99_ms"     : round(float(np.percentile(latencies_ms, 99)), 2),
    "ms_per_sample_mean"       : round(float(ms_per_sample.mean()), 2),
    "ms_per_token_mean"        : round(float(ms_per_token.mean()), 4),
    "ms_per_token_p50"         : round(float(np.median(ms_per_token)), 4),
    "tokens_per_sec_mean"      : round(float(tokens_per_sec.mean()), 1),
    "tokens_per_sec_p50"       : round(float(np.median(tokens_per_sec)), 1),
}

summary = {
    "timestamp"           : datetime.now().isoformat(),
    "model_path"          : os.path.abspath(model_path),
    "dataset"             : DATASET_ID,
    "split"               : SPLIT,
    "total_samples"       : len(valid_df),
    "overall_accuracy_pct": round(overall_acc, 4),
    "timing"              : timing_stats,
    "subject_accuracies"  : subject_acc["accuracy"].to_dict(),
}

tag = f"{INFERENCE_MODE}_bs{BATCH_SIZE}"
summary_path = os.path.join(OUTPUT_DIR, f"mmlu_redux_summary_{tag}.json")
detail_path  = os.path.join(OUTPUT_DIR, f"mmlu_redux_details_{tag}.csv")
subject_path = os.path.join(OUTPUT_DIR, f"mmlu_redux_by_subject_{tag}.csv")

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

df.to_csv(detail_path, index=False, encoding="utf-8-sig")
subject_acc.to_csv(subject_path, encoding="utf-8-sig")

print(f"[INFO] 결과 저장:")
print(f"  요약: {summary_path}")
print(f"  상세: {detail_path}")
print(f"  Subject별: {subject_path}")
print(f"\n{'█'*56}")
print(f"  MMLU-Redux 2.0 정확도 : {overall_acc:.2f}%")
print(f"  처리 속도             : {timing_stats['samples_per_sec']:.1f} samples/sec")
print(f"  토큰당 추론 시간       : {timing_stats['ms_per_token_mean']:.3f} ms/token")
print(f"  처리량                : {timing_stats['tokens_per_sec_mean']:.0f} tokens/sec")
print(f"{'█'*56}")

In [ ]:
try:
    import matplotlib.pyplot as plt
    import matplotlib
    matplotlib.rcParams["font.family"] = "DejaVu Sans"

    fig, axes = plt.subplots(2, 2, figsize=(16, 12))

    ax = axes[0, 0]
    ax.hist(subject_acc["accuracy"], bins=20, color="steelblue", edgecolor="white", alpha=0.85)
    ax.axvline(overall_acc, color="crimson", lw=2, ls="--", label=f"Overall: {overall_acc:.1f}%")
    ax.set(xlabel="Accuracy (%)", ylabel="Subject Count", title="Subject Accuracy Distribution")
    ax.legend(); ax.grid(True, alpha=0.3)

    ax = axes[0, 1]
    top_n = min(30, len(subject_acc))
    plot_data = pd.concat([subject_acc.head(top_n // 2), subject_acc.tail(top_n // 2)])
    colors = ["#2ecc71" if a >= 50 else "#e74c3c" for a in plot_data["accuracy"]]
    ax.barh(range(len(plot_data)), plot_data["accuracy"], color=colors, alpha=0.85)
    ax.set_yticks(range(len(plot_data)))
    ax.set_yticklabels(plot_data.index, fontsize=7)
    ax.axvline(overall_acc, color="crimson", lw=2, ls="--", label=f"Overall: {overall_acc:.1f}%")
    ax.set(xlabel="Accuracy (%)", title=f"Top/Bottom {top_n//2} Subjects", xlim=(0, 105))
    ax.legend(); ax.grid(True, alpha=0.3, axis="x")

    ax = axes[1, 0]
    ax.hist(latencies_ms, bins=30, color="darkorange", edgecolor="white", alpha=0.85)
    ax.axvline(latencies_ms.mean(), color="navy", lw=2, ls="--",
               label=f"Mean: {latencies_ms.mean():.0f} ms")
    ax.set(xlabel="Latency (ms)", ylabel="Count",
           title=f"Batch Latency (bs={BATCH_SIZE}, mode={INFERENCE_MODE})")
    ax.legend(); ax.grid(True, alpha=0.3)

    ax = axes[1, 1]
    ax.hist(ms_per_token, bins=30, color="mediumpurple", edgecolor="white", alpha=0.85)
    ax.axvline(ms_per_token.mean(), color="darkred", lw=2, ls="--",
               label=f"Mean: {ms_per_token.mean():.3f} ms/token")
    ax.set(xlabel="ms / token", ylabel="Count", title="Per-Token Latency")
    ax.legend(); ax.grid(True, alpha=0.3)

    plt.suptitle(
        f"MMLU-Redux 2.0 | {INFERENCE_MODE} mode, bs={BATCH_SIZE} | Acc={overall_acc:.1f}%",
        fontsize=13, fontweight="bold", y=1.01
    )
    plt.tight_layout()
    plot_path = os.path.join(OUTPUT_DIR, f"mmlu_results_{tag}.png")
    plt.savefig(plot_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"[INFO] 그래프 저장: {plot_path}")
except ImportError:
    print("[INFO] matplotlib 없음")
except Exception as e:
    print(f"[WARN] 시각화 오류: {e}")

In [ ]:
if _tmp_dir and os.path.exists(_tmp_dir):
    shutil.rmtree(_tmp_dir)
    print(f"[INFO] 임시 디렉토리 삭제: {_tmp_dir}")
print("[INFO] 평가 완료!")